In [1]:
# LCEL链式类型的表达
# 其允许输入类型集合对应输出类型集合，使用同样的方法调用组件
# 常见的方法invoke、stream、batch
# 优势：异步支持、内置批量和流式处理支持；可备用设置安全机制；内置日志；自动并行可并行分支

In [2]:
import json
# 设置环境
import os
import openai
from langchain_ollama import ChatOllama

openai.api_key = os.environ.get("DEEPSEEK_API_KEY")
openai.base_url = "https://api.deepseek.com/v1"

In [3]:
# need to pip install pydantic

In [4]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.schema.output_parser import StrOutputParser

In [5]:
prompt = ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)

model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    api_key=os.environ.get("DEEPSEEK_API_KEY") # 需要准备api_key
)

output_parser =StrOutputParser()

In [6]:
chain = prompt | model | output_parser

In [7]:
chain.invoke({"topic": "bears"})

'Why do bears have hairy coats?\n\nBecause they’d look weird in sweaters.'

In [8]:
# 创建更加复杂的链路
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_classic.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OllamaEmbeddings(model="qwen3-embedding:0.6b")
)

retriever = vectorstore.as_retriever()

In [9]:
retriever.invoke("where did harrison work?") # 在检索器上获取相关文档

[Document(metadata={}, page_content='harrison worked at kensho'),
 Document(metadata={}, page_content='bears like to eat honey')]

In [10]:
retriever.invoke("what does bear like to eat")

[Document(metadata={}, page_content='bears like to eat honey'),
 Document(metadata={}, page_content='harrison worked at kensho')]

In [11]:
# 上述操作可以在具有大量文档当中使用，并且返回最相关文档，其将被使用于检索增强生成管道
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [12]:
# 链的唯一输入是用户问题，创建接收单个问题处理
from langchain_classic.schema.runnable import RunnableMap

In [13]:
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
}) | prompt | model | output_parser

In [14]:
chain.invoke({"question":"where did harrison work?"})

'Harrison worked at Kensho.'

In [15]:
inputs = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
})

In [16]:
inputs.invoke({"question":"where did harrison work?"}) # 这两个元素会被传到Prompt当中，进而生成提示值，传递给模型，调用模型返回聊天信息，最后传输给解析器

{'context': [Document(metadata={}, page_content='harrison worked at kensho'),
  Document(metadata={}, page_content='bears like to eat honey')],
 'question': 'where did harrison work?'}

In [17]:
# 添加参数
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    temperature=0
).bind(functions=functions)

In [19]:
runnable = prompt | model

In [20]:
runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='I can’t give you the current live weather in San Francisco right now. For real-time conditions, check a weather service like:\n\n- **weather.com**\n- **National Weather Service** (weather.gov)\n- Your phone’s weather app\n\nIf you just want a general idea: San Francisco weather can vary by neighborhood and often includes **fog, cool temps (50s–60s°F), and possible wind**, especially in the summer mornings.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 89, 'total_tokens': 254, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 71, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 89}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-f

In [21]:
# 添加参数
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    },
    {
        "name": "sports_search",
        "description": "Search for new of recent sport events",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {
                    "type": "string",
                    "description": "The sports team to search for"
                },
            },
            "required": ["team_name"],
        }
    }
]

In [22]:
model = model.bind(functions=functions)

In [23]:
runnable = prompt | model

In [25]:
runnable.invoke({"input": "how did the patriots do yesterday?"})

AIMessage(content='The New England Patriots didn’t play yesterday—it’s the NFL offseason right now. Their 2025 season games are over, and the new season starts in September 2026.\n\nIf you mean a different team or a past game, let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 91, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 128, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 91}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '5732168a-ac22-48a8-86d2-68a414a54fa8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07ec5-5c7e-7c02-87

In [26]:
# Fallbacks
# from langchain_classic.llms import OpenAI
# import json

In [27]:
# simple_model = OpenAI( # 在早期过于简单的模型无法解析json输出，会出现json解码错误的情况
#     temperature=0,
#     max_tokens=1000,
#     model=""
# )

In [31]:
challenge = "writer three poems in a json blob, where each poem is a json blob of a title, author, and first line"

In [33]:
# 使用新的模型通过输出解析器创建链路
model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    model="deepseek-v4-flash",
    temperature=0
)

chain = model | StrOutputParser() | json.loads

In [35]:
chain.invoke(challenge)

[{'title': 'The Quiet Tide',
  'author': 'A. E. Lune',
  'first_line': 'The quiet tide rewrites the sand.'},
 {'title': 'Lanterns',
  'author': 'M. K. Night',
  'first_line': 'Lanterns drift down the river of night.'},
 {'title': 'Orchard Rain',
  'author': 'S. T. Willow',
  'first_line': 'Rain among the apple trees is a soft percussion.'}]

In [36]:
# final_chain = simple_chain.with_fallback([chain]) # 如果在simple_chain当中运行出现错误，那么就遍历chain这个列表，直至执行成功

NameError: name 'simple_chain' is not defined

In [38]:
# interface接口
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)

model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    model="deepseek-v4-flash",
)
output_parser = StrOutputParser()

chain = prompt | model | output_parser


In [39]:
chain.invoke({"topic": "bears"})

'Why do bears hate the internet?\n\nBecause they keep getting caught in bear traps.'

In [40]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

['What do you call a bear with no teeth?  \nA gummy bear!',
 'Why don’t frogs park on the road?  \nBecause they might get toad!']

In [41]:
for t in chain.stream({"topic": "bears"}):
    print(t)













Why
 do
 bears
 hate
 fast
 food
 restaurants
?


Because
 they
 can
’
t
 get
 a
 slow
 bear
-
ger
.




In [42]:
response = await chain.ainvoke({"topic": "bears"}) # 三个方法都支持异步处理
response

'Why do bears have hairy coats?  \nBecause they’d look silly in sweaters!'